# 03 — Encoding & Feature Engineering
**Lumen Data Science — Cross-Sell & Up-Sell Prediction**

Encodes categorical variables, engineers derived features (tenure, line revenue, GM%), and
builds customer-level aggregate statistics used by `cross_sell_model.ipynb`.

**Input:** `data/processed/data_clean_final.csv`
**Output:** `data/processed/customer_features.csv` — customer aggregates consumed by cross_sell_model

---
## 6. Categorical Encoding

For ML models we need numerical values. Two strategies are used:
- **Label encoding** for ordinal or high-cardinality variables
- **One-hot encoding** for low-cardinality variables (< 10 categories)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import os
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

print('Libraries loaded ✓')

Libraries loaded ✓


In [2]:
DATE_COLS = ['Invoice Date', 'Customer First Invoice Date']

df = pd.read_csv('../data/processed/data_clean_final.csv', low_memory=False, parse_dates=DATE_COLS)

df_encoded = df.copy()

df['Customer industry'].nunique()

15

In [3]:
df[['Sales Channel - Internal', 'Sales Channel - External', 'Manufacturing Location Code']].nunique()

Sales Channel - Internal       618
Sales Channel - External       618
Manufacturing Location Code     30
dtype: int64

In [4]:
df['Intercompany'].value_counts()

Intercompany
NO    928762
Name: count, dtype: int64

In [ ]:
# --- LABEL ENCODING (high cardinality or order doesn't matter) ---
label_encode_cols = [
    'CustomerID',
    'Item Code',
    'Manufacturing Location Code',
    'Sales Channel - Internal',
    'Sales Channel - External',
    'Top Customer Group',    # OTHER=0, STAR=1
    'Make vs Buy',           # BUY=0, MANUFACTURED=1
]

le_dict = {}  # save encoders for later inverse_transform

for col in label_encode_cols:
    if col in df_encoded.columns:
        le = LabelEncoder()
        df_encoded[col + '_enc'] = le.fit_transform(df_encoded[col].astype(str))
        le_dict[col] = le
        print(f'{col}: {df_encoded[col].nunique()} categories → label encoded')

In [ ]:
# --- ONE-HOT ENCODING (low cardinality) ---
ohe_cols = [
    'Manufacturing Region',   # North America, Asia, Europe
    'Customer Region',        # North America, Asia, Europe  
    'Product family',         # PF000, PF001, PF002
    'Customer industry',
]

for col in ohe_cols:
    if col in df_encoded.columns:
        dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=False)
        df_encoded = pd.concat([df_encoded, dummies], axis=1)
        print(f'{col}: one-hot → {dummies.shape[1]} new columns')

print(f'\nShape after encoding: {df_encoded.shape}')

In [8]:
print(df_encoded.columns.tolist())

['Manufacturing Region', 'Manufacturing Location Code', 'Intercompany', 'CustomerID', 'Customer industry', 'Customer Region', 'Customer First Invoice Date', 'Top Customer Group', 'Item Code', 'Product family', 'Product group', 'Make vs Buy', 'Sales Channel - Internal', 'Sales Channel - External', 'Invoice Date', 'Invoice #', 'Invoiced qty (shipped)', 'Ordered qty', 'Invoiced price', 'Invoiced price (TX)', 'Cost of part', 'GM%', 'CustomerID_enc', 'Item Code_enc', 'Manufacturing Location Code_enc', 'Sales Channel - Internal_enc', 'Sales Channel - External_enc', 'Top Customer Group_enc', 'Make vs Buy_enc', 'Manufacturing Region_Asia', 'Manufacturing Region_Europe', 'Manufacturing Region_North America', 'Manufacturing Region_Unknown', 'Customer Region_Asia', 'Customer Region_Europe', 'Customer Region_North America', 'Customer Region_Unknown', 'Product family_PF000', 'Product family_PF001', 'Product family_PF002', 'Customer industry_IC000', 'Customer industry_IC001', 'Customer industry_IC00

---
## 7. Feature Engineering — Derived Features

Key step — creating variables that give the model context about each customer's behaviour.

In [ ]:
# 7a. Temporal features from Invoice Date
df_encoded['invoice_year'] = df_encoded['Invoice Date'].dt.year
df_encoded['invoice_month'] = df_encoded['Invoice Date'].dt.month
df_encoded['invoice_quarter'] = df_encoded['Invoice Date'].dt.quarter
df_encoded['invoice_dayofweek'] = df_encoded['Invoice Date'].dt.dayofweek

print('Temporal features added ✓')

In [ ]:
# 7b. Customer tenure — how long the customer has been active (in days)
reference_date = df_encoded['Invoice Date'].max()
df_encoded['customer_tenure_days'] = (
    reference_date - df_encoded['Customer First Invoice Date']
).dt.days

print('Customer tenure computed ✓')
print(df_encoded['customer_tenure_days'].describe())

In [ ]:
# 7c. Line revenue (qty * price)
df_encoded['line_revenue'] = (
    df_encoded['Invoiced qty (shipped)'] * df_encoded['Invoiced price']
)

# 7d. Gross profit per line
df_encoded['line_gross_profit'] = df_encoded['line_revenue'] * df_encoded['GM%']

print('Revenue and gross profit features added ✓')

In [ ]:
# 7e. Customer-level aggregate features
# Core table for customer segmentation
customer_agg = df_encoded.groupby('CustomerID').agg(
    total_revenue=('line_revenue', 'sum'),
    total_orders=('Invoice #', 'nunique'),
    total_items=('Item Code', 'nunique'),
    avg_order_value=('line_revenue', 'mean'),
    avg_gm=('GM%', 'mean'),
    n_product_families=('Product family', 'nunique'),
    n_product_groups=('Product group', 'nunique'),
    customer_tenure_days=('customer_tenure_days', 'first'),
    customer_region=('Customer Region', 'first'),
    customer_industry=('Customer industry', 'first'),
    top_customer_group=('Top Customer Group', 'first'),
).reset_index()

print(f'Customer aggregate table: {customer_agg.shape}')
display(customer_agg.head())

In [ ]:
# 7f. Distribution plots of key features
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

customer_agg['total_revenue'].apply(lambda x: np.log1p(x)).hist(
    bins=40, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Log(Total Revenue) per Customer')

customer_agg['total_orders'].hist(bins=40, ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Orders per Customer')

customer_agg['total_items'].hist(bins=40, ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Unique Products per Customer')

customer_agg['avg_gm'].hist(bins=40, ax=axes[1,0], color='salmon')
axes[1,0].set_title('Avg GM% per Customer')

customer_agg['customer_tenure_days'].hist(bins=40, ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Customer Tenure (days)')

customer_agg['n_product_groups'].hist(bins=30, ax=axes[1,2], color='green', alpha=0.7)
axes[1,2].set_title('Product Groups per Customer')

plt.suptitle('Key Customer Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 7g. Product co-occurrence matrix (basis for cross-sell)
# Group by Invoice # — which products appear on the same invoice?
invoice_items = df_encoded.groupby('Invoice #')['Product group'].apply(
    lambda x: list(x.dropna().unique())
).reset_index()
invoice_items.columns = ['invoice_id', 'product_groups']

# Filter invoices with more than one product (cross-sell opportunities)
multi_product = invoice_items[invoice_items['product_groups'].apply(len) > 1]
print(f'Single-product invoices: {(invoice_items["product_groups"].apply(len) == 1).sum():,}')
print(f'Multi-product invoices (2+): {len(multi_product):,}')
print(f'% multi-product invoices: {len(multi_product)/len(invoice_items)*100:.1f}%')
display(multi_product.head(5))

---
## 8. Correlation Analysis of Numerical Features

In [ ]:
num_features = [
    'Invoiced qty (shipped)', 'Invoiced price', 'Cost of part',
    'GM%', 'line_revenue', 'customer_tenure_days',
    'invoice_month', 'invoice_quarter'
]

existing_num = [c for c in num_features if c in df_encoded.columns]
corr = df_encoded[existing_num].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

---
## 9. Final Dataset — Save for Modelling

In [ ]:
# Final overview
print('=== FINAL DATASET ===')
print(f'Shape: {df_encoded.shape}')
print(f'Nulls: {df_encoded.isnull().sum().sum()}')
print(f'\nColumns:')
for col in df_encoded.columns:
    print(f'  {col}: {df_encoded[col].dtype}')

In [17]:
df_encoded.isnull().sum()[df_encoded.isnull().sum() > 0]

Ordered qty          23170
Cost of part         19781
GM%                  19781
line_gross_profit    19781
dtype: int64

In [18]:
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)
print(f'Converted {len(bool_cols)} bool columns to int ✓')

Converted 26 bool columns to int ✓


In [ ]:
# Save processed datasets
df_encoded.to_csv('../data/processed/data_clean_enc_fe.csv', index=False)
customer_agg.to_csv('../data/processed/customer_features.csv', index=False)
invoice_items.to_csv('../data/processed/invoice_items.csv', index=False)

print('Saved ✓')
print('  → data/processed/data_clean_enc_fe.csv')
print('  → data/processed/customer_features.csv')
print('  → data/processed/invoice_items.csv')

---
## Summary

| Step | Action |
|------|--------|
| Encoding | Label encoding for high-cardinality, OHE for low-cardinality |
| New features | Tenure, revenue, gross profit, temporal features |
| Aggregates | Customer-level table for segmentation |
| Co-occurrence | Invoice-items table for market basket analysis |